# NYC Taxi Data Exploration

This notebook queries the Nessie Catalog (Iceberg) directly to check the Gold layer Data Marts.

In [3]:
from pyspark.sql import SparkSession
import pandas as pd
import os

# Hiển thị full chiều ngang cho Pandas
pd.set_option('display.max_columns', None)


# Khởi tạo Spark Session với Nessie Catalog (Nhánh Main)
spark = (
    SparkSession.builder
    .appName("data_exploration")
    .master("spark://spark-master:7077")
    .config("spark.driver.extraJavaOptions", "-Duser.dir=/tmp -Dlog4j2.configurationFile=/opt/spark/conf/log4j2.properties")
    .config("spark.eventLog.enabled", "false")\
    .getOrCreate()
)


spark.sparkContext.setLogLevel("ERROR")
print("Spark Session đã khởi tạo thành công!")
print(f"App ID: {spark.sparkContext.applicationId}")


26/07/05 09:00:49 WARN SparkContext: Another SparkContext is being constructed (or threw an exception in its constructor). This may indicate an error, since only one SparkContext should be running in this JVM (see SPARK-2243). The other SparkContext was created at:
org.apache.spark.api.java.JavaSparkContext.<init>(JavaSparkContext.scala:59)
java.base/jdk.internal.reflect.NativeConstructorAccessorImpl.newInstance0(Native Method)
java.base/jdk.internal.reflect.NativeConstructorAccessorImpl.newInstance(NativeConstructorAccessorImpl.java:77)
java.base/jdk.internal.reflect.DelegatingConstructorAccessorImpl.newInstance(DelegatingConstructorAccessorImpl.java:45)
java.base/java.lang.reflect.Constructor.newInstanceWithCaller(Constructor.java:500)
java.base/java.lang.reflect.Constructor.newInstance(Constructor.java:481)
py4j.reflection.MethodInvoker.invoke(MethodInvoker.java:247)
py4j.reflection.ReflectionEngine.invoke(ReflectionEngine.java:374)
py4j.Gateway.invoke(Gateway.java:238)
py4j.command

## 2. Check `revenue_by_zone` (Gold Layer)
View TOP 10 zones with the highest revenue.

In [4]:
print("--- TOP 10 Zones by Revenue ---")
df = spark.sql("""
    SELECT PULocationID, ZoneName, Borough, total_trips, total_revenue, avg_trip_distance
    FROM nessie.gold.revenue_by_zone 
    ORDER BY total_revenue DESC 
    LIMIT 10
""")
df.printSchema()
df.show(truncate=False)

--- TOP 10 Zones by Revenue ---
root
 |-- PULocationID: integer (nullable = true)
 |-- ZoneName: string (nullable = true)
 |-- Borough: string (nullable = true)
 |-- total_trips: long (nullable = true)
 |-- total_revenue: double (nullable = true)
 |-- avg_trip_distance: double (nullable = true)



[Stage 0:>                                                          (0 + 1) / 1]

+------------+----------------------------+---------+-----------+--------------------+------------------+
|PULocationID|ZoneName                    |Borough  |total_trips|total_revenue       |avg_trip_distance |
+------------+----------------------------+---------+-----------+--------------------+------------------+
|132         |JFK Airport                 |Queens   |136989     |1.1078013930002797E7|15.942973377424618|
|138         |LaGuardia Airport           |Queens   |86574      |5746372.020000111   |9.712479843833068 |
|161         |Midtown Center              |Manhattan|134975     |3234974.5399999376  |2.3179908131135307|
|237         |Upper East Side South       |Manhattan|135466     |2672962.5900000674  |1.704067293638257 |
|230         |Times Sq/Theatre District   |Manhattan|98970      |2658938.3699999363  |2.9592102657370725|
|236         |Upper East Side North       |Manhattan|128107     |2581848.4800000624  |1.844341370885278 |
|186         |Penn Station/Madison Sq West|Man

## 3. Check `daily_trips` (Gold Layer)
View trends for the last 5 days.

In [5]:
print("--- Last 5 Days ---")
df = spark.sql("""    
    SELECT trip_date, total_trips, total_revenue, avg_tip_percentage
    FROM nessie.gold.daily_trips 
    ORDER BY trip_date DESC 
    LIMIT 5
""")
df.printSchema()
df.show(truncate=False)

--- Last 5 Days ---
root
 |-- trip_date: date (nullable = true)
 |-- total_trips: long (nullable = true)
 |-- total_revenue: double (nullable = true)
 |-- avg_tip_percentage: double (nullable = true)

+----------+-----------+------------------+------------------+
|trip_date |total_trips|total_revenue     |avg_tip_percentage|
+----------+-----------+------------------+------------------+
|2024-01-31|95365      |2561465.2499999953|12.693203251817575|
|2024-01-30|94646      |2518880.109999988 |12.656952764871122|
|2024-01-29|78997      |2233295.17999993  |12.319804961191853|
|2024-01-28|82227      |2218143.249999927 |12.419575855191129|
|2024-01-27|102269     |2566227.1599999894|12.44394910754225 |
+----------+-----------+------------------+------------------+



## 4. Check `payment_type_summary` (Gold Layer)
Payment habits and Tip %.

In [6]:
print("--- Payment Habits ---")
df = spark.sql("""    
    SELECT payment_type, total_trips, total_revenue, avg_tip_percentage
    FROM nessie.gold.payment_type_summary
""")
df.printSchema()
df.show(truncate=False)

--- Payment Habits ---
root
 |-- payment_type: string (nullable = true)
 |-- total_trips: long (nullable = true)
 |-- total_revenue: double (nullable = true)
 |-- avg_tip_percentage: double (nullable = true)

+------------+-----------+-------------------+--------------------+
|payment_type|total_trips|total_revenue      |avg_tip_percentage  |
+------------+-----------+-------------------+--------------------+
|Dispute     |22716      |568675.7100000054  |0.01113513267621264 |
|No charge   |9884       |220342.46000000119 |0.017573880193356498|
|Credit card |2273558    |6.388902937003414E7|14.798112303003819  |
|Cash        |417894     |9981610.670001073  |5.704586465490034E-4|
+------------+-----------+-------------------+--------------------+



## 5. Check `monthly_summary` (Gold Layer)
View aggregated metrics by month.

In [7]:
print("--- Monthly Summary ---")
df = spark.sql("""    
    SELECT Year, Month, total_trips, total_revenue, avg_trip_duration_seconds
    FROM nessie.gold.monthly_summary
    ORDER BY Year DESC, Month DESC
    LIMIT 12
""")
df.printSchema()
df.show(truncate=False)

--- Monthly Summary ---
root
 |-- Year: integer (nullable = true)
 |-- Month: integer (nullable = true)
 |-- total_trips: long (nullable = true)
 |-- total_revenue: double (nullable = true)
 |-- avg_trip_duration_seconds: double (nullable = true)

+----+-----+-----------+-------------------+-------------------------+
|Year|Month|total_trips|total_revenue      |avg_trip_duration_seconds|
+----+-----+-----------+-------------------+-------------------------+
|2024|1    |2724052    |7.465965821001375E7|945.1229800312182        |
+----+-----+-----------+-------------------+-------------------------+

